# LeetCode 1323 — Maximum 69 Number bằng Python

**Link đề:** https://leetcode.com/problems/maximum-69-number/
**Độ khó:** Easy · **Chủ đề:** Math, Greedy (tham lam theo vị trí chữ số)

```python
class Solution:
    def maximum69Number (self, num: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **giá trị vị trí** của chữ số: chữ số càng bên trái càng "nặng".
2. Từ brute force (thử đổi từng chữ số) rút ra **quy tắc tham lam**: đổi chữ số `6` **đầu tiên từ trái**.
3. Thành thạo công cụ: `str`, `list(str)`, `''.join`, `str.replace(old, new, count)`, `str.find`, `% 10`, `// 10`, `**`.
4. Có 4 lời giải: brute force, `replace` 1 lần, duyệt chuỗi, thuần toán (cộng `3 × 10^k`).
5. Chứng minh tham lam đúng, Time/Space, lỗi hay gặp, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

*Lưu ý:* skeleton của LeetCode có một dấu cách trước `(` trong `maximum69Number (self, ...)` — Python chấp nhận, ta giữ nguyên tên method.

## 1. Đề bài

Cho số nguyên dương `num` chỉ gồm các chữ số `6` và `9`.

Trả về số **lớn nhất** có thể nhận được bằng cách thay **nhiều nhất một** chữ số (`6` thành `9`, hoặc `9` thành `6`).

| `num` | Output | Giải thích |
|---:|---:|---|
| `9669` | `9969` | các lựa chọn: 6669, **9969**, 9699, 9666 (hoặc không đổi 9669) |
| `9996` | `9999` | đổi chữ số `6` cuối |
| `9999` | `9999` | không đổi gì là tốt nhất |

**Constraints:** `1 <= num <= 10^4`, `num` chỉ gồm chữ số `6` và `9`.

Câu hỏi trước khi code:
- Đổi `9` → `6` có bao giờ làm số lớn hơn không?
- Nếu có nhiều chữ số `6`, nên đổi cái nào?

## 2. Nền tảng: giá trị vị trí của chữ số

Số `9669` thực chất là:

```text
9669 = 9·1000 + 6·100 + 6·10 + 9·1
```

Đổi một chữ số `6` thành `9` ở hàng `10^k` làm số **tăng đúng `3·10^k`**:
- đổi hàng trăm (k = 2): `+300` → `9969`
- đổi hàng chục (k = 1): `+30` → `9699`

Vậy hàng càng cao, mức tăng càng lớn.

In [1]:
num = 9669
print("9·1000 + 6·100 + 6·10 + 9·1 =", 9 * 1000 + 6 * 100 + 6 * 10 + 9 * 1)
print("đổi hàng trăm  :", num + 3 * 10 ** 2)
print("đổi hàng chục  :", num + 3 * 10 ** 1)
print("10 ** 3 =", 10 ** 3, "(toán tử ** là lũy thừa)")

9·1000 + 6·100 + 6·10 + 9·1 = 9669
đổi hàng trăm  : 9969
đổi hàng chục  : 9699
10 ** 3 = 1000 (toán tử ** là lũy thừa)


### 2.1. Chuỗi ↔ số, và chuỗi là bất biến

- `str(9669)` → `"9669"`, `int("9969")` → `9969`.
- Chuỗi **không sửa trực tiếp** được: `s[1] = '9'` báo lỗi `TypeError`. Muốn sửa phải đổi sang `list`, sửa, rồi `''.join`.

In [2]:
s = str(9669)
try:
    s[1] = '9'
except TypeError as e:
    print("Không sửa được chuỗi:", e)

chars = list(s)          # ['9', '6', '6', '9']
chars[1] = '9'
print(chars, "->", int(''.join(chars)))

Không sửa được chuỗi: 'str' object does not support item assignment
['9', '9', '6', '9'] -> 9969


### 2.2. `str.replace(old, new, count)`

`s.replace('6', '9')` thay **tất cả**; thêm tham số thứ ba `count = 1` thì chỉ thay **lần xuất hiện đầu tiên (từ trái)**.

In [3]:
s = "9669"
print(s.replace('6', '9'))       # thay hết
print(s.replace('6', '9', 1))    # chỉ thay cái đầu tiên
print("9999".replace('6', '9', 1), "<- không có 6 thì giữ nguyên")

9999
9969
9999 <- không có 6 thì giữ nguyên


### 2.3. `str.find(sub)` — vị trí xuất hiện đầu tiên

`"9669".find('6')` → `1`. Không tìm thấy thì trả `-1` (khác với `index()` sẽ báo lỗi).

In [4]:
print("9669".find('6'), "9996".find('6'), "9999".find('6'))
try:
    "9999".index('6')
except ValueError as e:
    print("index() báo lỗi:", e)

1 3 -1
index() báo lỗi: substring not found


### 2.4. So sánh hai số cùng độ dài: chữ số trái nhất khác nhau quyết định

Với hai số **cùng số chữ số**, so sánh giống tra từ điển: đi từ trái sang, vị trí đầu tiên khác nhau quyết định số nào lớn hơn. Ví dụ `9969` vs `9699`: hai chữ số đầu `9 = 9`, chữ số thứ hai `9 > 6` ⇒ `9969` lớn hơn — **bất kể** các chữ số phía sau.

Với chuỗi cùng độ dài chỉ gồm chữ số, so sánh chuỗi `"9969" > "9699"` cho cùng kết quả với so sánh số. (Khác độ dài thì **không** được: `"10" < "9"` là `True` vì so ký tự `'1' < '9'`.)

In [5]:
print(9969 > 9699, "9969" > "9699")
print("khác độ dài -> so chuỗi sai:", "10" < "9", "| so số:", 10 < 9)
a, c = "9969", "9699"
for i, (x, y) in enumerate(zip(a, c)):
    print(f"vị trí {i}: {x} vs {y}", "-> khác nhau, quyết định!" if x != y else "")
    if x != y:
        break

True True
khác độ dài -> so chuỗi sai: True | so số: False
vị trí 0: 9 vs 9 
vị trí 1: 9 vs 6 -> khác nhau, quyết định!


## 3. Cách trực giác / baseline — thử mọi lựa chọn

Đề cho phép "nhiều nhất một" lần đổi. Có `d + 1` lựa chọn (đổi một trong `d` chữ số, hoặc không đổi). Ta thử hết và lấy max — đúng như hint của đề.

In [6]:
def max69_brute(num: int) -> int:
    s = list(str(num))
    best = num                       # lựa chọn "không đổi"
    for i in range(len(s)):
        original = s[i]
        s[i] = '9' if original == '6' else '6'   # lật chữ số i
        best = max(best, int(''.join(s)))
        s[i] = original              # trả lại như cũ
    return best

for num in [9669, 9996, 9999, 6, 9, 6666]:
    print(num, "->", max69_brute(num))

9669 -> 9969
9996 -> 9999
9999 -> 9999
6 -> 9
9 -> 9
6666 -> 9666


### 3.1. In ra tất cả lựa chọn để "nhìn" quy luật

In [7]:
def show_all_options(num: int) -> None:
    s = list(str(num))
    print(f"num = {num} (không đổi)")
    for i in range(len(s)):
        t = s.copy()                 # copy để không làm hỏng s
        t[i] = '9' if t[i] == '6' else '6'
        print(f"  đổi vị trí {i} ({s[i]} -> {t[i]}): {''.join(t)}")

show_all_options(9669)
print()
show_all_options(6699)

num = 9669 (không đổi)
  đổi vị trí 0 (9 -> 6): 6669
  đổi vị trí 1 (6 -> 9): 9969
  đổi vị trí 2 (6 -> 9): 9699
  đổi vị trí 3 (9 -> 6): 9666

num = 6699 (không đổi)
  đổi vị trí 0 (6 -> 9): 9699
  đổi vị trí 1 (6 -> 9): 6999
  đổi vị trí 2 (9 -> 6): 6669
  đổi vị trí 3 (9 -> 6): 6696


Quan sát:
- Đổi `9 → 6` luôn làm **nhỏ đi** → không bao giờ có lợi.
- Trong các lần đổi `6 → 9`, lần đổi ở vị trí **trái nhất** cho số lớn nhất.

Brute force tốn `O(d²)` (d lựa chọn × dựng lại chuỗi `O(d)`). Với `num <= 10^4` thì `d <= 5`, vẫn nhanh. Nhưng ta muốn hiểu **vì sao** để có lời giải `O(d)` gọn gàng.

## 4. Insight — tham lam: đổi chữ số `6` đầu tiên từ trái

**Quy tắc:** tìm chữ số `6` **đầu tiên tính từ trái**, đổi nó thành `9`. Nếu không có `6`, giữ nguyên.

Lý do (sẽ chứng minh kỹ ở mục sau): đổi `6 → 9` ở hàng `10^k` tăng `3·10^k`; hàng trái nhất có `6` là hàng **cao nhất** có thể tăng; và `3·10^k` lớn hơn mọi mức tăng ở các hàng thấp hơn.

In [8]:
for k in range(4):
    print(f"đổi ở hàng 10^{k}: tăng {3 * 10 ** k}")
print("3·10^2 =", 3 * 10 ** 2, "> tổng mọi mức tăng hàng thấp hơn 3·(10+1) =", 3 * (10 + 1))

đổi ở hàng 10^0: tăng 3
đổi ở hàng 10^1: tăng 30
đổi ở hàng 10^2: tăng 300
đổi ở hàng 10^3: tăng 3000
3·10^2 = 300 > tổng mọi mức tăng hàng thấp hơn 3·(10+1) = 33


## 5. Lời giải chính — `replace` đúng 1 lần

**Đoạn 1:** đổi `num` sang chuỗi.

**Đoạn 2:** `replace('6', '9', 1)` — chỉ thay `6` đầu tiên từ trái (chính là quy tắc tham lam).

**Đoạn 3:** đổi lại thành `int`.

In [9]:
def max69_replace(num: int) -> int:
    return int(str(num).replace('6', '9', 1))

for num in [9669, 9996, 9999, 6, 9, 6666, 6999]:
    print(num, "->", max69_replace(num))

9669 -> 9969
9996 -> 9999
9999 -> 9999
6 -> 9
9 -> 9
6666 -> 9666
6999 -> 9999


### 5.1. Cùng ý tưởng, viết tường minh bằng vòng lặp (cho người mới)

In [10]:
def max69_loop(num: int) -> int:
    chars = list(str(num))
    for i in range(len(chars)):
        if chars[i] == '6':
            chars[i] = '9'
            break                    # chỉ đổi MỘT lần rồi dừng
    return int(''.join(chars))

for num in [9669, 9996, 9999]:
    print(num, "->", max69_loop(num))

9669 -> 9969
9996 -> 9999
9999 -> 9999


`break` thoát ngay khỏi vòng `for` gần nhất. Không có `break` thì ta sẽ đổi **mọi** chữ số `6` — vi phạm "nhiều nhất một".

## 6. TRACE từng bước

Trace `num = 9669` với `max69_loop`:

| i | chars[i] | là '6'? | hành động | chars sau bước |
|---|---|---|---|---|
| 0 | `9` | không | bỏ qua | `9 6 6 9` |
| 1 | `6` | **có** | đổi thành `9`, `break` | `9 9 6 9` |

Kết quả `9969`. Vị trí `2` (cũng là `6`) **không** được xét vì đã `break`.

In [11]:
def trace_max69(num: int) -> int:
    chars = list(str(num))
    print("num =", num, "| chars =", chars)
    for i in range(len(chars)):
        if chars[i] == '6':
            chars[i] = '9'
            print(f"  i={i}: gặp '6' -> đổi thành '9', chars={chars}, dừng (break)")
            break
        print(f"  i={i}: '{chars[i]}' không phải 6, đi tiếp")
    else:
        print("  không có chữ số 6 nào -> giữ nguyên")
    result = int(''.join(chars))
    print("  kết quả:", result)
    return result

trace_max69(9669)
print()
trace_max69(9996)
print()
trace_max69(9999)

num = 9669 | chars = ['9', '6', '6', '9']
  i=0: '9' không phải 6, đi tiếp
  i=1: gặp '6' -> đổi thành '9', chars=['9', '9', '6', '9'], dừng (break)
  kết quả: 9969

num = 9996 | chars = ['9', '9', '9', '6']
  i=0: '9' không phải 6, đi tiếp
  i=1: '9' không phải 6, đi tiếp
  i=2: '9' không phải 6, đi tiếp
  i=3: gặp '6' -> đổi thành '9', chars=['9', '9', '9', '9'], dừng (break)
  kết quả: 9999

num = 9999 | chars = ['9', '9', '9', '9']
  i=0: '9' không phải 6, đi tiếp
  i=1: '9' không phải 6, đi tiếp
  i=2: '9' không phải 6, đi tiếp
  i=3: '9' không phải 6, đi tiếp
  không có chữ số 6 nào -> giữ nguyên
  kết quả: 9999


9999

Ở đây dùng cấu trúc ít gặp **`for ... else`**: khối `else` chạy khi vòng `for` kết thúc **mà không gặp `break`** — tức là không tìm thấy `6`.

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def maximum69Number (self, num: int) -> int:
        return int(str(num).replace('6', '9', 1))

for num in [9669, 9996, 9999]:
    print(num, Solution().maximum69Number(num))

9669 9969
9996 9999
9999 9999


## 8. Các lời giải khác

### Cách 3 — thuần toán: tìm hàng cao nhất có chữ số 6, cộng `3·10^k`

Duyệt chữ số từ **phải sang trái** bằng `% 10` / `// 10`, ghi nhớ vị trí `k` **lớn nhất** có chữ số `6` (vì đi từ phải sang, lần gặp **sau cùng** là hàng cao nhất). Cuối cùng cộng `3 * 10**k`.

In [13]:
def max69_math(num: int) -> int:
    x = num
    k = 0                 # hàng hiện tại
    highest_six = -1      # chưa thấy 6
    while x > 0:
        if x % 10 == 6:
            highest_six = k
        x //= 10
        k += 1
    if highest_six == -1:
        return num
    return num + 3 * 10 ** highest_six

for num in [9669, 9996, 9999, 6, 6666]:
    print(num, "->", max69_math(num))

9669 -> 9969
9996 -> 9999
9999 -> 9999
6 -> 9
6666 -> 9666


In [14]:
def trace_max69_math(num: int) -> int:
    x, k, highest_six = num, 0, -1
    print("num =", num)
    while x > 0:
        digit = x % 10
        if digit == 6:
            highest_six = k
        print(f"  hàng 10^{k}: chữ số {digit} -> highest_six = {highest_six}")
        x //= 10
        k += 1
    add = 3 * 10 ** highest_six if highest_six >= 0 else 0
    print(f"  cộng thêm {add} -> {num + add}")
    return num + add

trace_max69_math(9669)

num = 9669
  hàng 10^0: chữ số 9 -> highest_six = -1
  hàng 10^1: chữ số 6 -> highest_six = 1
  hàng 10^2: chữ số 6 -> highest_six = 2
  hàng 10^3: chữ số 9 -> highest_six = 2
  cộng thêm 300 -> 9969


9969

### Cách 4 — `find` vị trí rồi tính bằng toán

`i = s.find('6')` cho vị trí từ trái (0-based). Với chuỗi dài `d`, vị trí `i` từ trái ứng với hàng `10^(d-1-i)`.

In [15]:
def max69_find(num: int) -> int:
    s = str(num)
    i = s.find('6')
    if i == -1:
        return num
    return num + 3 * 10 ** (len(s) - 1 - i)

for num in [9669, 9996, 9999, 6]:
    print(num, "->", max69_find(num))

9669 -> 9969
9996 -> 9999
9999 -> 9999
6 -> 9


### Code nộp LeetCode cho các cách khác

In [16]:
class SolutionBrute:
    def maximum69Number (self, num: int) -> int:
        s = list(str(num))
        best = num
        for i in range(len(s)):
            old = s[i]
            s[i] = '9' if old == '6' else '6'
            best = max(best, int(''.join(s)))
            s[i] = old
        return best

class SolutionMath:
    def maximum69Number (self, num: int) -> int:
        x, k, high = num, 0, -1
        while x:
            if x % 10 == 6:
                high = k
            x //= 10
            k += 1
        return num if high < 0 else num + 3 * 10 ** high

class SolutionFind:
    def maximum69Number (self, num: int) -> int:
        s = str(num)
        i = s.find('6')
        return num if i < 0 else num + 3 * 10 ** (len(s) - 1 - i)

for cls in [SolutionBrute, SolutionMath, SolutionFind]:
    print(cls.__name__, [cls().maximum69Number(x) for x in [9669, 9996, 9999]])

SolutionBrute [9969, 9999, 9999]
SolutionMath [9969, 9999, 9999]
SolutionFind [9969, 9999, 9999]


### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | thử lật từng chữ số, lấy max | `O(d²)` | `O(d)` | không cần hiểu tham lam |
| 2. `replace(..., 1)` | đổi 6 đầu tiên | `O(d)` | `O(d)` | ngắn nhất, nên dùng |
| 2b. Vòng lặp + `break` | như trên, tường minh | `O(d)` | `O(d)` | dễ giải thích |
| 3. Thuần toán | hàng cao nhất có 6, cộng `3·10^k` | `O(d)` | `O(1)` | không dùng chuỗi |
| 4. `find` + toán | vị trí → hàng | `O(d)` | `O(d)` | kết hợp |

`d` = số chữ số của `num` (≤ 5).

## 9. Vì sao tham lam đúng?

Gọi `k` là hàng cao nhất chứa chữ số `6`.

1. **Đổi `9 → 6` không bao giờ có lợi:** làm số giảm `3·10^j`. Vì "không đổi" cũng là lựa chọn hợp lệ, ta bỏ qua mọi phép `9 → 6`.
2. **Đổi `6 → 9` ở hàng `j` tăng đúng `3·10^j`.** Chỉ được đổi một lần, nên ta muốn `j` **lớn nhất** có thể, tức `j = k`.
3. Mọi `j < k` cho mức tăng `3·10^j < 3·10^k`. Vậy đổi ở hàng `k` là tối ưu.
4. Nếu không có `6`: mọi phép đổi đều làm giảm → giữ nguyên.

Hàng cao nhất = vị trí **trái nhất** trong chuỗi ⇒ `replace('6', '9', 1)` (thay từ trái) làm đúng việc này. ∎

### 9.1. Độ phức tạp

`d` = số chữ số = `O(log10 num)`.
- Cách `replace`/`find`/vòng lặp: Time `O(d)`, Space `O(d)` (chuỗi tạm).
- Cách thuần toán: Time `O(d)`, Space `O(1)`.
- Brute force: Time `O(d²)`.

Với constraints `num <= 10^4`, mọi cách đều tức thì.

## 10. Bộ test

Test mẫu đề, biên (1 chữ số `6`, `9`), toàn `6`, toàn `9`, và **đối chiếu** các cách với brute force trên **mọi** số chỉ gồm `6`/`9` có 1–5 chữ số.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.maximum69Number(9669), 9969)
check("ví dụ 2", sol.maximum69Number(9996), 9999)
check("ví dụ 3", sol.maximum69Number(9999), 9999)
check("một chữ số 6", sol.maximum69Number(6), 9)
check("một chữ số 9", sol.maximum69Number(9), 9)
check("toàn 6", sol.maximum69Number(6666), 9666)
check("6 ở đầu", sol.maximum69Number(6999), 9999)

ví dụ 1        PASS -> 9969
ví dụ 2        PASS -> 9999
ví dụ 3        PASS -> 9999
một chữ số 6   PASS -> 9
một chữ số 9   PASS -> 9
toàn 6         PASS -> 9666
6 ở đầu        PASS -> 9999


In [18]:
from itertools import product

all_nums = [int(''.join(p)) for d in range(1, 6) for p in product('69', repeat=d)]
print("số lượng input cần kiểm:", len(all_nums), "| vài số:", all_nums[:6])
methods = [max69_replace, max69_loop, max69_math, max69_find,
           SolutionMath().maximum69Number, SolutionFind().maximum69Number, sol.maximum69Number]
for num in all_nums:
    expected = max69_brute(num)
    for f in methods:
        assert f(num) == expected, (f, num)
print("Mọi cách khớp brute force trên toàn bộ", len(all_nums), "input")

số lượng input cần kiểm: 62 | vài số: [6, 9, 66, 69, 96, 99]
Mọi cách khớp brute force trên toàn bộ 62 input


In [19]:
# Kiểm tra tính chất của đáp án (property test):
# (1) kết quả >= num, (2) khác num ở nhiều nhất 1 chữ số, (3) kết quả chỉ gồm 6 và 9
for num in all_nums:
    r = sol.maximum69Number(num)
    diff = sum(1 for x, y in zip(str(num), str(r)) if x != y)
    assert r >= num and diff <= 1 and set(str(r)) <= {'6', '9'}
print("3 tính chất đúng cho cả", len(all_nums), "input")
print("set(...) <= {...} nghĩa là 'tập con':", {'9'} <= {'6', '9'}, {'7'} <= {'6', '9'})

3 tính chất đúng cho cả 62 input
set(...) <= {...} nghĩa là 'tập con': True False


`itertools.product('69', repeat=d)` sinh mọi tổ hợp độ dài `d` từ các ký tự `'6'`, `'9'` — giống `d` vòng `for` lồng nhau. Có `2 + 4 + 8 + 16 + 32 = 62` số.

## 11. Lỗi người mới hay gặp

### Lỗi 1: Thay tất cả chữ số `6`

In [20]:
def wrong_replace_all(num):
    return int(str(num).replace('6', '9'))

print("sai :", wrong_replace_all(9669), "| đúng:", max69_replace(9669))

sai : 9999 | đúng: 9969


**Sửa:** thêm `count = 1`, hoặc `break` sau lần đổi đầu.

### Lỗi 2: Đổi chữ số `6` cuối cùng (từ phải)

Khi dùng cách toán duyệt từ phải sang, có người `break` ngay khi gặp `6` đầu tiên — đó là `6` ở **hàng thấp nhất**.

In [21]:
def wrong_lowest_six(num):
    x, k = num, 0
    while x > 0:
        if x % 10 == 6:
            return num + 3 * 10 ** k     # sai: dừng ở 6 thấp nhất
        x //= 10
        k += 1
    return num

print("sai :", wrong_lowest_six(9669), "| đúng:", max69_math(9669))

sai : 9699 | đúng: 9969


### Lỗi 3: Quên đổi kết quả về `int`

`str(num).replace(...)` trả về **chuỗi** `"9969"`; LeetCode yêu cầu `int`. So sánh `"9969" == 9969` là `False`.

### Lỗi 4: Lệch hàng khi dùng `find`

Vị trí `i` từ trái ứng với hàng `10^(len(s) - 1 - i)`, không phải `10^i`.

In [22]:
print(repr(str(9669).replace('6', '9', 1)), "==", 9969, "?", str(9669).replace('6', '9', 1) == 9969)
s = "9669"; i = s.find('6')
print("sai hàng 10^i       :", 9669 + 3 * 10 ** i)
print("đúng hàng 10^(d-1-i):", 9669 + 3 * 10 ** (len(s) - 1 - i))

'9969' == 9969 ? False
sai hàng 10^i       : 9699
đúng hàng 10^(d-1-i): 9969


## 12. Mẹo nhận dạng pattern

- "Đổi **nhiều nhất một** chữ số để được số lớn nhất/nhỏ nhất" ⇒ **tham lam từ chữ số cao nhất** (trái nhất).
- So sánh số có cùng độ dài ⇒ so sánh từ trái sang phải (thứ tự từ điển); chữ số trái quyết định.
- Thay đổi chữ số ở hàng `10^k` thay đổi giá trị `Δ·10^k` ⇒ dùng được cả bằng chuỗi và bằng toán.
- Brute force trên `O(d)` lựa chọn là cách tốt để **kiểm chứng** quy tắc tham lam.

## 13. Biến thể

1. **Số nhỏ nhất** khi được đổi nhiều nhất một chữ số: đổi `9` đầu tiên thành `6`.
2. **Được đổi tối đa `k` chữ số**: đổi `k` chữ số `6` đầu tiên (`replace('6', '9', k)`).
3. **LeetCode 1432 — Max Difference You Can Get From Changing an Integer**, **2566 — Maximum Difference by Remapping a Digit**: cùng tư duy "chữ số trái nhất quyết định".

In [23]:
def min69(num: int) -> int:
    return int(str(num).replace('9', '6', 1))

def max69_k(num: int, k: int) -> int:
    return int(str(num).replace('6', '9', k))

print("nhỏ nhất của 9669:", min69(9669))
print("đổi tối đa 2 lần, 6666:", max69_k(6666, 2))

nhỏ nhất của 9669: 6669
đổi tối đa 2 lần, 6666: 9966


## 14. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Liệt kê mọi lựa chọn của `6969` và chọn max.

**Bài 2 (đoán output):** `"66".replace('6', '9', 1)`, `"99".find('6')`, `3 * 10 ** 0`.

**Bài 3 (tự viết):** `exercise_max69(num)` bằng vòng lặp + `break` (không dùng `replace`).

**Bài 4 (thuần toán):** `exercise_max69_math(num)` không dùng chuỗi.

**Bài 5 (biến thể):** `exercise_min_digit_change(num)` — `num` chỉ gồm `6` và `9`, trả về số **nhỏ nhất** sau tối đa một lần đổi.

**Bài 6 (mini challenge):** `exercise_max_diff(num)` — `num` chỉ gồm `6` và `9`; trả về hiệu lớn nhất giữa số lớn nhất và số nhỏ nhất đạt được (mỗi số được tạo bằng tối đa một lần đổi, độc lập nhau).

In [24]:
def exercise_max69(num: int) -> int:
    # TODO: list(str(num)), tìm '6' đầu tiên, đổi, break
    pass

def exercise_max69_math(num: int) -> int:
    # TODO: % 10, // 10, ghi nhớ hàng cao nhất có 6
    pass

# assert exercise_max69(9669) == 9969 and exercise_max69(9999) == 9999
# assert exercise_max69_math(9669) == 9969 and exercise_max69_math(6) == 9

In [25]:
def exercise_min_digit_change(num: int) -> int:
    # TODO
    pass

def exercise_max_diff(num: int) -> int:
    # TODO
    pass

# assert exercise_min_digit_change(9669) == 6669
# assert exercise_min_digit_change(6666) == 6666
# assert exercise_max_diff(9669) == 9969 - 6669

## 15. Gợi ý 3 mức

**Bài 1** — (1) 4 vị trí + "không đổi". (2) Lật từng vị trí. (3) Max là đổi vị trí 0.

**Bài 3** — (1) Chuỗi không sửa được → `list`. (2) `if chars[i] == '6'`. (3) Đổi rồi `break`, cuối cùng `int(''.join(chars))`.

**Bài 4** — (1) Duyệt từ phải sang, biến `k` đếm hàng. (2) Gặp 6 thì cập nhật `high = k` (không dừng). (3) Trả `num + 3 * 10 ** high` nếu `high >= 0`.

**Bài 5** — (1) Muốn nhỏ nhất thì giảm ở hàng cao nhất. (2) Đổi `9` trái nhất thành `6`. (3) `replace('9', '6', 1)`.

**Bài 6** — (1) Hai bài con độc lập. (2) Dùng lại bài 3 và bài 5. (3) Trả `max - min`.

## 16. Đáp án tham khảo

**Bài 1:** `6969` (không đổi), `9969`, `6669`, `6999`, `6966` → max `9969`.

**Bài 2:** `"96"`, `-1`, `3`.

In [26]:
def answer_max69(num):
    chars = list(str(num))
    for i in range(len(chars)):
        if chars[i] == '6':
            chars[i] = '9'
            break
    return int(''.join(chars))

def answer_max69_math(num):
    x, k, high = num, 0, -1
    while x > 0:
        if x % 10 == 6:
            high = k
        x //= 10
        k += 1
    return num if high < 0 else num + 3 * 10 ** high

def answer_min_digit_change(num):
    return int(str(num).replace('9', '6', 1))

def answer_max_diff(num):
    return answer_max69(num) - answer_min_digit_change(num)

assert max69_brute(6969) == 9969
assert "66".replace('6', '9', 1) == "96" and "99".find('6') == -1 and 3 * 10 ** 0 == 3
for num in all_nums:
    assert answer_max69(num) == max69_brute(num) == answer_max69_math(num)
    s = list(str(num)); best_min = num
    for i in range(len(s)):
        t = s.copy(); t[i] = '9' if t[i] == '6' else '6'
        best_min = min(best_min, int(''.join(t)))
    assert answer_min_digit_change(num) == best_min
assert answer_min_digit_change(9669) == 6669 and answer_min_digit_change(6666) == 6666
assert answer_max_diff(9669) == 3300
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maximum69Number (self, num: int) -> int:
        return int(str(num).replace('6', '9', 1))   # chỉ thay 6 TRÁI NHẤT
```

Ý tưởng một câu: **đổi chữ số 6 ở hàng cao nhất thành 9** (tăng `3·10^k`, lớn nhất có thể).

**Checklist:**
- [ ] Chỉ đổi **một** chữ số (`count = 1` hoặc `break`).
- [ ] Đổi chữ số **trái nhất**, không phải phải nhất.
- [ ] Trả về `int`, không phải `str`.
- [ ] Đã test `9999` (không đổi) và `6` (một chữ số).